# 01 - First Look at the Data (Omar, S1.12)

This notebook takes a first look at the processed sessions: how many are ready,
their shapes, neurons per brain region, choice balance, block counts, and the
error and zero-contrast trials. It uses the team modules
(`src.common.settings`, `src.analysis.eda`). No model is trained here.

## Plan

1. Load the configuration and list the processed sessions.
2. Summarise every ready session in one table.
3. Look at one session in detail: neurons per brain region.
4. Check the choice balance, the block prior and the control trials.
5. Look at the distribution of firing rates across neurons.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside the brain-decoding repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis.eda import block_counts, region_counts, summarize_sessions
from src.common.settings import load_config, list_processed_sessions, load_session

config = load_config()
print("Project:", config["project_name"])
print("Feature window (seconds):", config["data"]["window"])

## 2. Which processed sessions are ready

In [ ]:
ready = list(list_processed_sessions())
print("Processed sessions available:", len(ready))

if not ready:
    raise SystemExit(
        "No processed sessions found. "
        "Run: python -m src.data.build_dataset"
    )

sessions = [load_session(sid) for sid in ready]

## 3. Summary of every session

In [ ]:
summary = summarize_sessions(sessions)
display(summary)

## 4. One session in detail

In [ ]:
data = sessions[0]
x = np.asarray(data["X"], dtype=float)
print("Session:", str(data["session_id"]))
print("X shape (trials x neurons):", x.shape)
print("Fields:", sorted(data))

### Neurons per brain region

In [ ]:
rc = region_counts(data)
display(rc.rename("neurons").to_frame())

fig, ax = plt.subplots(figsize=(8, 3.5))
rc.plot.bar(ax=ax)
ax.set_ylabel("Number of neurons")
ax.set_title("Neurons per brain region")
plt.tight_layout()
plt.show()

### Choice balance, block prior and control trials

In [ ]:
y = np.asarray(data["y"])
print("Trials:", len(y))
print("Choice balance (mean of y):", round(float(y.mean()), 3))
print("Error trials:", int(np.sum(data["is_error"])))
print("Zero-contrast trials:", int(np.sum(data["is_zero_contrast"])))
display(block_counts(data).rename("trials").to_frame())

### Firing rates

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(x.mean(axis=0), bins=20)
ax.set_xlabel("Mean firing rate per neuron (Hz)")
ax.set_ylabel("Number of neurons")
ax.set_title("Firing-rate distribution")
plt.tight_layout()
plt.show()

## Conclusions

No result is claimed in this notebook. Every number above comes from the
processed sessions built by S1.7. Re-run the notebook whenever the session
list changes.